<a href="https://colab.research.google.com/github/addamit/py-howtos/blob/master/ClusteringConversations_AdaptiveSize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install sentence-transformers
# !pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 42.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [1]:
import numpy as np
import json
import logging
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ==============================
# Memory Management Utilities
# ==============================

def estimate_token_count(text, tokenizer):
    """Estimate the number of tokens in a text string."""
    tokens = tokenizer.encode(text)
    return len(tokens)

def check_gpu_memory():
    """Check and log available GPU memory."""
    if torch.cuda.is_available():
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        reserved_memory = torch.cuda.memory_reserved(0) / 1024**3
        allocated_memory = torch.cuda.memory_allocated(0) / 1024**3
        free_memory = total_memory - reserved_memory

        logger.info(f"GPU Memory: Total={total_memory:.2f}GB, "
                    f"Reserved={reserved_memory:.2f}GB, "
                    f"Allocated={allocated_memory:.2f}GB, "
                    f"Free={free_memory:.2f}GB")
        return free_memory
    else:
        logger.info("No GPU available")
        return None

def adaptive_sample_selection(cluster_indices, cluster_map, tokenizer, max_tokens=2000):
    """
    Adaptively select samples from a cluster based on token count limits.

    Args:
        cluster_indices: List of indices of clusters to sample from
        cluster_map: Current cluster mapping dictionary
        tokenizer: The tokenizer to use for estimating token counts
        max_tokens: Maximum tokens to allow for each of in/out samples

    Returns:
        in_samples and out_samples strings with token counts under the limit
    """
    # Start timing for performance monitoring
    start_time = time.time()

    # Use beginning indices for in-samples and end indices for out-samples
    mid_point = len(cluster_indices) // 2
    in_cluster_indices = cluster_indices[:mid_point]
    out_cluster_indices = cluster_indices[mid_point:]

    # Get the original cluster map keys as a list for indexing
    cluster_keys = list(cluster_map.keys())

    # Start with a reasonable number of samples
    initial_sample_count = min(10, len(in_cluster_indices), len(out_cluster_indices))

    # Initialize with small number of samples
    in_samples = [cluster_map[cluster_keys[idx]]["name"] for idx in in_cluster_indices[:initial_sample_count]]
    out_samples = [cluster_map[cluster_keys[idx]]["name"] for idx in out_cluster_indices[:initial_sample_count]]

    # Convert to strings with sample separators
    in_samples_str = "\n- " + "\n- ".join(in_samples)
    out_samples_str = "\n- " + "\n- ".join(out_samples)

    # Check token counts
    in_token_count = estimate_token_count(in_samples_str, tokenizer)
    out_token_count = estimate_token_count(out_samples_str, tokenizer)

    logger.debug(f"Initial samples - In: {initial_sample_count} samples, {in_token_count} tokens | "
                f"Out: {initial_sample_count} samples, {out_token_count} tokens")

    # Try to add more samples if we're under the token limit
    i = initial_sample_count
    while (i < len(in_cluster_indices) and
           in_token_count < max_tokens and
           i < len(out_cluster_indices) and
           out_token_count < max_tokens):

        # Try adding an in-sample
        if i < len(in_cluster_indices):
            new_sample = cluster_map[cluster_keys[in_cluster_indices[i]]]["name"]
            new_sample_tokens = estimate_token_count("\n- " + new_sample, tokenizer)

            if in_token_count + new_sample_tokens < max_tokens:
                in_samples.append(new_sample)
                in_samples_str += "\n- " + new_sample
                in_token_count += new_sample_tokens
                logger.debug(f"Added in-sample #{i+1}, new token count: {in_token_count}")
            else:
                logger.debug(f"Stopped adding in-samples at #{i}, would exceed token limit")

        # Try adding an out-sample
        if i < len(out_cluster_indices):
            new_sample = cluster_map[cluster_keys[out_cluster_indices[i]]]["name"]
            new_sample_tokens = estimate_token_count("\n- " + new_sample, tokenizer)

            if out_token_count + new_sample_tokens < max_tokens:
                out_samples.append(new_sample)
                out_samples_str += "\n- " + new_sample
                out_token_count += new_sample_tokens
                logger.debug(f"Added out-sample #{i+1}, new token count: {out_token_count}")
            else:
                logger.debug(f"Stopped adding out-samples at #{i}, would exceed token limit")

        i += 1

    # Log the final sample selection stats
    elapsed_time = time.time() - start_time
    logger.info(f"Sample selection complete in {elapsed_time:.2f}s - "
                f"In: {len(in_samples)} samples, {in_token_count} tokens | "
                f"Out: {len(out_samples)} samples, {out_token_count} tokens")

    return in_samples_str, out_samples_str

# ==============================
# Data Loading and Preparation
# ==============================

def load_data(sample_size=1000):
    """Load sample data from the C4 dataset."""
    # dataset = load_dataset("allenai/c4", split="train", streaming=True)
    # texts = [row["text"] for row in dataset.take(sample_size)]
    dataset = load_dataset("determined-ai/consumer_complaints_medium", split="train")
    texts = [sample for sample in dataset.select(range(200))]  # First 200 convos

    return texts


def load_data2(sample_size=1000):
    """Load sample data from the C4 dataset."""
    logger.info(f"Loading {sample_size} samples from C4 dataset...")
    start_time = time.time()

    dataset = load_dataset("allenai/c4", split="train", streaming=True)
    texts = [row["text"] for row in dataset.take(sample_size)]

    elapsed_time = time.time() - start_time
    logger.info(f"Data loading complete in {elapsed_time:.2f}s - {len(texts)} texts loaded")

    return texts

def create_embeddings(texts, model_name="all-mpnet-base-v2"):
    """Convert texts to embeddings using a sentence transformer model."""
    logger.info(f"Creating embeddings using {model_name}...")
    start_time = time.time()

    # Check GPU memory before loading model
    check_gpu_memory()

    model = SentenceTransformer(model_name)

    # Generate embeddings in batches to manage memory
    batch_size = 32
    total_batches = (len(texts) + batch_size - 1) // batch_size

    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        logger.debug(f"Embedding batch {i//batch_size + 1}/{total_batches} with {len(batch_texts)} texts")

        batch_embeddings = model.encode(batch_texts, normalize_embeddings=True)
        all_embeddings.append(batch_embeddings)

        if (i//batch_size + 1) % 10 == 0:
            logger.info(f"Completed {i//batch_size + 1}/{total_batches} embedding batches")
            check_gpu_memory()

    embeddings = np.vstack(all_embeddings)

    elapsed_time = time.time() - start_time
    logger.info(f"Embedding creation complete in {elapsed_time:.2f}s - Shape: {embeddings.shape}")

    return model, embeddings

# ==============================
# Clustering Utilities
# ==============================

def calculate_similarity_matrix(embeddings):
    """Calculate the cosine similarity between all pairs of embeddings."""
    logger.info(f"Calculating similarity matrix for {len(embeddings)} embeddings...")
    start_time = time.time()

    # For large matrices, calculate in chunks to save memory
    if len(embeddings) > 1000:
        logger.info("Large embedding set detected, using chunked similarity calculation")
        chunk_size = 500
        num_chunks = (len(embeddings) + chunk_size - 1) // chunk_size
        similarity_matrix = np.zeros((len(embeddings), len(embeddings)))

        for i in range(num_chunks):
            start_i = i * chunk_size
            end_i = min((i + 1) * chunk_size, len(embeddings))
            chunk_i = embeddings[start_i:end_i]

            for j in range(i, num_chunks):
                start_j = j * chunk_size
                end_j = min((j + 1) * chunk_size, len(embeddings))
                chunk_j = embeddings[start_j:end_j]

                logger.debug(f"Calculating similarity for chunk ({i},{j}) of {num_chunks}x{num_chunks}")
                sim_chunk = cosine_similarity(chunk_i, chunk_j)

                similarity_matrix[start_i:end_i, start_j:end_j] = sim_chunk
                if i != j:  # Mirror the matrix for symmetry
                    similarity_matrix[start_j:end_j, start_i:end_i] = sim_chunk.T
    else:
        similarity_matrix = cosine_similarity(embeddings)

    elapsed_time = time.time() - start_time
    logger.info(f"Similarity matrix calculation complete in {elapsed_time:.2f}s - Shape: {similarity_matrix.shape}")

    return similarity_matrix

def create_neighborhoods(embeddings, avg_clusters_per_neighborhood=40):
    """Create initial neighborhoods using K-means clustering."""
    num_neighborhoods = max(1, round(len(embeddings) / avg_clusters_per_neighborhood))
    logger.info(f"Creating {num_neighborhoods} neighborhoods using K-means...")
    start_time = time.time()

    kmeans = KMeans(n_clusters=num_neighborhoods, random_state=42, n_init=10)
    neighborhood_labels = kmeans.fit_predict(embeddings)

    # Group indices by neighborhood
    neighborhoods = {i: [] for i in range(num_neighborhoods)}
    for idx, label in enumerate(neighborhood_labels):
        neighborhoods[label].append(idx)

    # Log neighborhood statistics
    neighborhood_sizes = [len(clusters) for clusters in neighborhoods.values()]
    avg_size = sum(neighborhood_sizes) / len(neighborhoods)
    min_size = min(neighborhood_sizes)
    max_size = max(neighborhood_sizes)

    elapsed_time = time.time() - start_time
    logger.info(f"Neighborhood creation complete in {elapsed_time:.2f}s - "
                f"Avg size: {avg_size:.1f}, Min: {min_size}, Max: {max_size}")

    return neighborhoods

def expand_neighborhoods(neighborhoods, similarity_matrix, m_nearest=5):
    """
    Expand each neighborhood by including the M nearest clusters from other neighborhoods.
    """
    logger.info(f"Expanding {len(neighborhoods)} neighborhoods with {m_nearest} nearest neighbors...")
    start_time = time.time()

    expanded_neighborhoods = {}

    for n_id, cluster_indices in neighborhoods.items():
        # Get indices of clusters not in this neighborhood
        other_clusters = list(set(range(len(similarity_matrix))) - set(cluster_indices))
        external_nearest = []

        # For each cluster in this neighborhood, find the M nearest from other neighborhoods
        for idx in cluster_indices:
            similarities = [(other_idx, similarity_matrix[idx][other_idx]) for other_idx in other_clusters]
            similarities.sort(key=lambda x: x[1], reverse=True)
            external_nearest.extend([s[0] for s in similarities[:m_nearest]])

        # Add the nearest external clusters to this neighborhood, removing duplicates
        expanded_neighborhoods[n_id] = list(set(cluster_indices + external_nearest))

        logger.debug(f"Neighborhood {n_id}: Original size: {len(cluster_indices)}, "
                    f"Expanded size: {len(expanded_neighborhoods[n_id])}")

    # Calculate expansion statistics
    original_sizes = [len(clusters) for clusters in neighborhoods.values()]
    expanded_sizes = [len(clusters) for clusters in expanded_neighborhoods.values()]
    avg_expansion = sum(expanded_sizes) / sum(original_sizes)

    elapsed_time = time.time() - start_time
    logger.info(f"Neighborhood expansion complete in {elapsed_time:.2f}s - "
                f"Average expansion factor: {avg_expansion:.2f}x")

    return expanded_neighborhoods

# ==============================
# Cluster Naming with Local Model
# ==============================

def setup_local_phi_model(model_path="microsoft/phi-3-mini-4k-instruct"):
    """Set up the local Phi model and tokenizer."""
    logger.info(f"Loading local model: {model_path}...")
    start_time = time.time()

    # Check GPU memory before loading model
    check_gpu_memory()

    # Load in 4-bit quantization to reduce memory usage
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_path)

        # Try loading with lower precision
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,  # Use half precision
            device_map="auto",
            load_in_4bit=True  # Use 4-bit quantization
        )

        logger.info("Successfully loaded model with 4-bit quantization")
    except Exception as e:
        logger.warning(f"Error loading model with 4-bit quantization: {e}")
        logger.info("Falling back to standard loading")

        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto"
        )

    # Check memory after loading
    check_gpu_memory()

    elapsed_time = time.time() - start_time
    logger.info(f"Model loading complete in {elapsed_time:.2f}s")

    return model, tokenizer

def generate_cluster_name_description_local(in_cluster_samples, out_cluster_samples, model, tokenizer, max_new_tokens=200):
    """
    Uses local Phi model to generate a name and description for a cluster.

    Args:
        in_cluster_samples: String with samples from within the cluster
        out_cluster_samples: String with samples from outside the cluster
        model: The local Phi model
        tokenizer: The tokenizer for the model
        max_new_tokens: Maximum tokens to generate

    Returns:
        Generated cluster name and description
    """
    logger.info("Generating cluster name and description with local model...")
    start_time = time.time()

    # Check memory before inference
    check_gpu_memory()

    # Create the prompt
    prompt = (
        f"<|system|>\n"
        f"You are a helpful AI assistant that can analyze text and identify patterns.\n"
        f"</s>\n"
        f"<|user|>\n"
        f"Below are two sets of text summaries:\n\n"
        f"**In-cluster summaries:**\n{in_cluster_samples}\n\n"
        f"**Out-cluster summaries:**\n{out_cluster_samples}\n\n"
        f"Generate a short but descriptive name (1-5 words) and a brief summary (2-3 sentences) that captures the theme "
        f"of the in-cluster summaries while clearly distinguishing them from the out-cluster summaries.\n"
        f"</s>\n"
        f"<|assistant|>\n"
    )

    # Log token usage
    input_tokens = tokenizer.encode(prompt)
    logger.info(f"Input token count: {len(input_tokens)}")

    try:
        # Generate the response
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                top_p=0.9,
                do_sample=True
            )

        # Decode the response
        full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract just the assistant's response
        response = full_response.split("<|assistant|>")[-1].strip()

        # Log token usage
        output_tokens = tokenizer.encode(response)
        logger.info(f"Output token count: {len(output_tokens)}")

    except Exception as e:
        logger.error(f"Error during generation: {e}")
        # Provide a fallback response
        response = "Generic Cluster\n\nThis cluster contains related content."

    elapsed_time = time.time() - start_time
    logger.info(f"Name generation complete in {elapsed_time:.2f}s - Response length: {len(response)} chars")

    # Check memory after inference
    check_gpu_memory()

    return response

def parse_cluster_info(cluster_info_text):
    """Parse the cluster name and description from the model's response."""
    lines = cluster_info_text.split("\n")
    name = lines[0].strip()

    # If name is too long, truncate it
    if len(name) > 50:
        name = name[:47] + "..."

    description = "\n".join(lines[1:]).strip() if len(lines) > 1 else ""

    return name, description

# ==============================
# Hierarchical Clustering
# ==============================

def perform_one_level_clustering(cluster_map, embedding_model, phi_model, tokenizer,
                                avg_clusters_per_neighborhood=40, m_nearest=5, max_token_per_sample=1000):
    """
    Perform one level of hierarchical clustering with memory-efficient sample selection.
    """
    level_start_time = time.time()
    logger.info(f"Starting clustering level with {len(cluster_map)} clusters")

    # Get cluster names and create embeddings
    cluster_texts = [cluster_map[i]["name"] for i in cluster_map.keys()]

    logger.info(f"Creating embeddings for {len(cluster_texts)} cluster texts")
    cluster_embeddings = embedding_model.encode(cluster_texts, normalize_embeddings=True)

    # Create initial neighborhoods using K-means
    neighborhoods = create_neighborhoods(cluster_embeddings, avg_clusters_per_neighborhood)

    # Calculate similarity between all clusters
    similarity_matrix = calculate_similarity_matrix(cluster_embeddings)

    # Expand neighborhoods to include nearby clusters
    expanded_neighborhoods = expand_neighborhoods(neighborhoods, similarity_matrix, m_nearest)

    # Create new cluster map by merging clusters in each neighborhood
    new_cluster_map = {}
    total_neighborhoods = len(expanded_neighborhoods)

    for idx, (n_id, cluster_indices) in enumerate(expanded_neighborhoods.items()):
        logger.info(f"Processing neighborhood {idx+1}/{total_neighborhoods} with {len(cluster_indices)} clusters")

        # Adaptively select samples within token limits
        in_cluster_samples, out_cluster_samples = adaptive_sample_selection(
            cluster_indices, cluster_map, tokenizer, max_token_per_sample)

        # Generate cluster name and description using local Phi model
        cluster_info = generate_cluster_name_description_local(
            in_cluster_samples, out_cluster_samples, phi_model, tokenizer)

        name, description = parse_cluster_info(cluster_info)
        logger.info(f"Generated cluster name: '{name}'")

        # Create new cluster and add children
        new_cluster_map[n_id] = {"name": name, "description": description, "children": {}}
        for idx in cluster_indices:
            old_cluster_id = list(cluster_map.keys())[idx]
            new_cluster_map[n_id]["children"][old_cluster_id] = cluster_map[old_cluster_id]

    level_elapsed_time = time.time() - level_start_time
    reduction_factor = len(cluster_map) / len(new_cluster_map) if len(new_cluster_map) > 0 else float('inf')
    logger.info(f"Clustering level complete in {level_elapsed_time:.2f}s - "
                f"Reduced from {len(cluster_map)} to {len(new_cluster_map)} clusters "
                f"(reduction factor: {reduction_factor:.2f}x)")

    return new_cluster_map

def hierarchical_clustering(embeddings, texts, target_clusters, model_path="microsoft/phi-3-mini-4k-instruct",
                           avg_clusters_per_neighborhood=40, m_nearest=5, max_token_per_sample=1000):
    """
    Performs memory-efficient hierarchical clustering with local model for naming.
    """
    total_start_time = time.time()
    logger.info(f"Starting hierarchical clustering process to reach {target_clusters} target clusters")

    # Initialize embedding model
    embedding_model = SentenceTransformer("all-mpnet-base-v2")

    # Initialize the local Phi model
    phi_model, tokenizer = setup_local_phi_model(model_path)

    # Initialize with one document per cluster
    level = 0
    cluster_map = {i: {"name": texts[i], "children": {}} for i in range(len(texts))}

    logger.info(f"Initial cluster count: {len(cluster_map)}")

    # Keep clustering until we reach the target number of clusters
    while len(cluster_map) > target_clusters:
        level += 1
        logger.info(f"\n==== Level {level}: Clustering {len(cluster_map)} clusters ====")

        # Perform one level of clustering with adaptive sample selection
        cluster_map = perform_one_level_clustering(
            cluster_map,
            embedding_model,
            phi_model,
            tokenizer,
            avg_clusters_per_neighborhood,
            m_nearest,
            max_token_per_sample
        )

        # If we can't reduce further, break
        if level > 1 and len(cluster_map) >= prev_cluster_count:
            logger.warning(f"Could not reduce cluster count further after {level} levels")
            break

        prev_cluster_count = len(cluster_map)

        # Save intermediate results
        save_hierarchy(cluster_map, f"cluster_hierarchy_level_{level}.json")

    total_elapsed_time = time.time() - total_start_time
    logger.info(f"Hierarchical clustering complete in {total_elapsed_time:.2f}s - "
                f"Final cluster count: {len(cluster_map)}")

    return cluster_map

# ==============================
# Exploration and Saving
# ==============================

def save_hierarchy(hierarchy, filename="cluster_hierarchy.json"):
    """Save the cluster hierarchy to a JSON file."""
    logger.info(f"Saving hierarchy to {filename}")
    with open(filename, "w") as f:
        json.dump(hierarchy, f, indent=4)
    logger.info(f"Hierarchy saved successfully")

def explore_clusters(cluster_hierarchy, depth=0):
    """Interactive exploration of the hierarchical clusters."""
    while True:
        print("\n" + "-" * (depth * 2) + " Available Clusters " + "-" * (depth * 2))
        for i, cluster_id in enumerate(cluster_hierarchy.keys()):
            print(f"{i}: {cluster_hierarchy[cluster_id]['name']} - {cluster_hierarchy[cluster_id].get('description', '')}")

        choice = input("\nEnter number to zoom in, 'b' to go back, or 'q' to quit: ")
        if choice == "q":
            break
        elif choice == "b":
            return
        elif choice.isdigit() and int(choice) in range(len(cluster_hierarchy)):
            selected_cluster = list(cluster_hierarchy.keys())[int(choice)]
            if cluster_hierarchy[selected_cluster]["children"]:
                explore_clusters(cluster_hierarchy[selected_cluster]["children"], depth + 1)
            else:
                print("\nNo further subclusters available.")

# ==============================
# Main Function
# ==============================

def main(sample_size=100, target_clusters=5, model_path="microsoft/phi-3-mini-4k-instruct",
         avg_clusters_per_neighborhood=20, m_nearest=3, max_token_per_sample=800):
    """
    Run the complete hierarchical clustering process with memory optimization.

    Args:
        sample_size: Number of samples to use (smaller value = less memory)
        target_clusters: Final number of top-level clusters
        model_path: Path to the local Phi model
        avg_clusters_per_neighborhood: Controls neighborhood size (smaller = less memory)
        m_nearest: Number of nearest neighbors to include (smaller = less memory)
        max_token_per_sample: Maximum tokens for in/out samples (smaller = less memory)
    """
    logger.info("=" * 80)
    logger.info("STARTING HIERARCHICAL CLUSTERING WITH MEMORY OPTIMIZATION")
    logger.info("=" * 80)
    logger.info(f"Parameters: sample_size={sample_size}, target_clusters={target_clusters}, "
                f"avg_clusters_per_neighborhood={avg_clusters_per_neighborhood}, "
                f"m_nearest={m_nearest}, max_token_per_sample={max_token_per_sample}")

    # Load and prepare data - use a reduced sample size
    texts = load_data(sample_size)

    # Create embeddings for the texts
    _, embeddings = create_embeddings(texts)

    # Perform hierarchical clustering with memory optimization
    final_hierarchy = hierarchical_clustering(
        embeddings,
        texts,
        target_clusters,
        model_path,
        avg_clusters_per_neighborhood,
        m_nearest,
        max_token_per_sample
    )

    # Save results
    save_hierarchy(final_hierarchy, "final_cluster_hierarchy.json")

    # Explore clusters
    logger.info("Starting interactive cluster exploration")
    explore_clusters(final_hierarchy)



In [3]:
# Use smaller values for all parameters to save memory
main(
    sample_size=100,            # Reduced from 1000
    target_clusters=3,          # Reduced from 5
    avg_clusters_per_neighborhood=20,  # Reduced from 40
    m_nearest=3,                # Reduced from 5
    max_token_per_sample=800    # Control token count sent to model
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

TypeError: sequence item 0: expected str instance, dict found

In [1]:
import numpy as np
import pandas as pd
import json
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


In [3]:
# Load dataset
dataset = load_dataset("determined-ai/consumer_complaints_medium", split="train")
conversations = [sample for sample in dataset.select(range(200))]  # First 200 convos


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/606 [00:00<?, ?B/s]

(…)-00000-of-00001-f6ccf6490f123247.parquet:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

(…)-00000-of-00001-0b18a4667c074099.parquet:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/64292 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/21439 [00:00<?, ? examples/s]

In [4]:
df = pd.json_normalize(conversations)

In [5]:

df.head()

,Issue,Consumer Complaint
0,Incorrect information on credit report,I have not been living at the address which ju...
1,"Managing, opening, or closing account",I have the Rush card and have not been able to...
2,Problems caused by my funds being low,Suntrust covered a debit card transaction when...
3,Incorrect information on credit report,I wrote Equifax on XXXX/XXXX/15 and alerted th...
4,Problem with a credit reporting company's inve...,"Over two months ago, I asked Equifax to invest..."


In [6]:
titles, docs = list(df['Issue']), list(df['Consumer Complaint'])

In [7]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [70]:
import torch
# Delete the model and tokenizer
del model
del tokenizer
del pipe
# Clear the cache
torch.cuda.empty_cache()

# Trigger garbage collection
gc.collect()

NameError: name 'model' is not defined

SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False}) with Transformer model: MPNetModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [8]:
model_path = "microsoft/Phi-3.5-mini-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_path)



config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py:   0%|          | 0.00/73.8k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [9]:
# Create a text generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)


Device set to use cuda:0


In [10]:
# Check for context size attributes
if hasattr(model.config, 'max_position_embeddings'):
    print(f"Maximum sequence length : {model.config.max_position_embeddings}")
elif hasattr(model.config, 'n_positions'):
    print(f"Maximum sequence length: {model.config.n_positions}")
else:
    print("Context size attribute not found.")

Maximum sequence length : 131072


In [82]:

def estimate_token_count(texts):
    """Estimate total tokens in a list of texts."""
    return sum(len(tokenizer.encode(t)) for t in texts)

In [11]:
def generate_cluster_name_description(in_cluster_samples, out_cluster_samples):
    """
    Uses the HF model to generate a name and description that defines
    the in-cluster samples while differentiating them from out-cluster samples.

    Args:
        in_cluster_samples (list of str): 50 summaries from within the cluster.
        out_cluster_samples (list of str): 50 summaries from outside but near the cluster.

    Returns:
        dict: {"name": str, "description": str}
    """
    prompt = (
        f"Below are two sets of text summaries:\n\n"
        f"**In-cluster summaries:**\n{in_cluster_samples}\n\n"
        f"**Out-cluster summaries:**\n{out_cluster_samples}\n\n"
        f"Generate a short but descriptive name and a summary that captures the theme of "
        f"the in-cluster summaries while clearly distinguishing them from the out-cluster summaries."
    )

    # Define the messages for the pipeline
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt},
    ]

    # Define generation arguments
    generation_args = {
        "max_new_tokens": 250,
        "return_full_text": False,
        "temperature": 1.0,
        "do_sample": True,
    }

    # Generate the response
    output = pipe(messages, **generation_args)
    response = output[0]['generated_text']

    # Extract name and description from the response
    # Assuming the response format is consistent and can be split into name and description
    name = "Generated Name"  # Placeholder for name extraction logic
    description = response

    return {"name": name, "description": description}


In [12]:
embeddings = embedding_model.encode(docs, batch_size=32, show_progress_bar=True) #, normalize_embeddings=True)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [13]:

embeddings.shape

(200, 768)

In [14]:
TARGET_CLUSTER_COUNT = 5
M_NEAREST = 5
AVG_CLUSTERS_PER_NEIGHBORHOOD = 40

In [15]:
level = 0
cluster_map = {i: {"name": docs[i], "children": {}} for i in range(len(docs))}
cluster_texts = [cluster_map[i]["name"] for i in cluster_map.keys()]
cluster_embeddings = embedding_model.encode(cluster_texts, normalize_embeddings=True)

In [17]:
len(cluster_map), len(cluster_texts), len(cluster_embeddings)

(200, 200, 200)

In [18]:
cluster_map[100]

{'name': 'Several times I have tried for free report and have been told I must download forms and send in requested information. I have answered ID question honestly and still told I have to do the mail in process. Why?',
 'children': {}}

In [34]:
num_neighborhoods = max(1, round(len(cluster_texts) / AVG_CLUSTERS_PER_NEIGHBORHOOD))

In [35]:
num_neighborhoods

5

In [36]:
kmeans = KMeans(n_clusters=num_neighborhoods, random_state=42, n_init=10)
neighborhood_labels = kmeans.fit_predict(cluster_embeddings)

In [39]:
neighborhoods = {i: [] for i in range(num_neighborhoods)}
for idx, label in enumerate(neighborhood_labels):
    neighborhoods[label].append(idx)

In [42]:
similarity_matrix = cosine_similarity(cluster_embeddings)

In [47]:
expanded_neighborhoods = {}
for n_id, cluster_indices in neighborhoods.items():

      other_clusters = list(set(range(len(cluster_embeddings))) - set(cluster_indices))

      external_nearest = []

      for idx in cluster_indices:
          similarities = [(other_idx, similarity_matrix[idx][other_idx]) for other_idx in other_clusters]
          similarities.sort(key=lambda x: x[1], reverse=True)
          external_nearest.extend([s[0] for s in similarities[:M_NEAREST]])

      expanded_neighborhoods[n_id] = list(set(cluster_indices + external_nearest))


In [52]:
len(expanded_neighborhoods[2]), len(neighborhoods[2])

(110, 40)

In [53]:
new_cluster_map = {}
for n_id, cluster_indices in expanded_neighborhoods.items():
    in_cluster_samples = [cluster_map[list(cluster_map.keys())[idx]]["name"] for idx in cluster_indices[:50]]
    out_cluster_samples = [cluster_map[list(cluster_map.keys())[idx]]["name"] for idx in cluster_indices[-50:]]

    # cluster_info = generate_cluster_name_description(in_cluster_samples, out_cluster_samples)
    # new_cluster_name = cluster_info.split("\n")[0]
    # new_cluster_description = "\n".join(cluster_info.split("\n")[1:])

    # new_cluster_map[n_id] = {"name": new_cluster_name, "description": new_cluster_description, "children": {}}
    # for idx in cluster_indices:
    #     old_cluster_id = list(cluster_map.keys())[idx]
    #     new_cluster_map[n_id]["children"][old_cluster_id] = cluster_map[old_cluster_id]


In [65]:
test_in_cluster_samples, test_out_cluster_samples = in_cluster_samples[0], out_cluster_samples[0]
len(test_in_cluster_samples), len(test_out_cluster_samples)

(155, 218)

In [67]:
test_in_cluster_samples, test_out_cluster_samples

("I have the Rush card and have not been able to access my money for weeks. My bills have not been paid at home and my daughter college funds ca n't be paid.",
 'I am XXXX. I have been struggle to meet the payments since I have XXXX job that I have been working and still struggle to make the full payment. I have call and it take long time because I was using the relay operator.')

In [74]:
generate_cluster_name_description(test_in_cluster_samples, test_out_cluster_samples)

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


{'name': 'Generated Name',
 'description': ' Name: Financial Strain and Access Issues\n\nSummary:\nThis theme encompasses individuals confronting critical financial strain due to unpaid bills and compromised access to funds, with a distinct divergence from out-cluster peers who primarily struggle with making payments due to lower income levels and prolonged communication delays associated with relay services. The in-cluster situation underscores the urgency of unresolved financial obligations that impact essential areas such as housing maintenance and educational expenses, whereas the out-cluster focuses on the challenges posed by systemic inefficiencies in service interactions.'}

In [75]:
test_in_cluster_samples, test_out_cluster_samples = in_cluster_samples[10], out_cluster_samples[10]
len(test_in_cluster_samples), len(test_out_cluster_samples)

(437, 359)

In [76]:
test_in_cluster_samples, test_out_cluster_samples

('i like fia and do not want to do this. FIA card services i asked for credit line increrase on card ending XXXX was declined for reasons not accurate. i have not asked before now. and i do pay. i was granted XXXX on another card recently. with it this way i will not use it much. these reasons are inaccurate. i ask for someone to check it since i can not wait on hold all day i wanted XXXX line now there is a hit on my credit card line.',
 'I sold my home last year and BBVA COMPASS BANK reported BANKRUPTCY on my credit report on XX/XX/XXXX. I have contacted COMPASS BANK numerous times and no one has returned my phone call. I have had to hire a credit company to try and resolve this issue. We sold our home in XX/XX/XXXX and purchase our new home in XX/XX/XXXX. I have never filed for Bankruptcy.')

In [77]:
generate_cluster_name_description(test_in_cluster_samples, test_out_cluster_samples)

{'name': 'Generated Name',
 'description': " **Name:** Credit Line Adjustment Concerns and Customer Service Experiences\n\n**Summary:**\nThe in-cluster summaries revolve around individuals experiencing difficulties in adjusting their existing credit lines with FIA card services, including denied requests due to alleged inaccuracies and frustrations with the response time and lack of assistance. In contrast, the out-cluster summaries detail a separate issue where an individual experiences repeated difficulties with COMPASS BANK reporting a bankruptcy on their credit report, despite the customer's claims of never filing for bankruptcy and lack of communication from the bank's representatives. These accounts share themes of credit management and customer service struggles but address distinctly different financial concerns."}

In [84]:
estimate_token_count(test_in_cluster_samples)

437

In [20]:
import logging
import torch

In [21]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [22]:
def estimate_token_count(text, tokenizer):
    """Estimate the number of tokens in a text string."""
    tokens = tokenizer.encode(text)
    return len(tokens)

def check_gpu_memory():
    """Check and log available GPU memory."""
    if torch.cuda.is_available():
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        reserved_memory = torch.cuda.memory_reserved(0) / 1024**3
        allocated_memory = torch.cuda.memory_allocated(0) / 1024**3
        free_memory = total_memory - reserved_memory

        logger.info(f"GPU Memory: Total={total_memory:.2f}GB, "
                    f"Reserved={reserved_memory:.2f}GB, "
                    f"Allocated={allocated_memory:.2f}GB, "
                    f"Free={free_memory:.2f}GB")
        return free_memory
    else:
        logger.info("No GPU available")
        return None

In [23]:
check_gpu_memory()

0.84674072265625

In [2]:
# ==============================
# Data Loading and Preparation
# ==============================

def load_data(sample_size=1000):
    """Load sample data from the C4 dataset."""
    # dataset = load_dataset("allenai/c4", split="train", streaming=True)
    # texts = [row["text"] for row in dataset.take(sample_size)]
    dataset = load_dataset("determined-ai/consumer_complaints_medium", split="train")
    texts = [sample for sample in dataset.select(range(200))]  # First 200 convos

    return texts

def create_embeddings(texts, model_name="all-mpnet-base-v2"):
    """Convert texts to embeddings using a sentence transformer model."""
    model = SentenceTransformer(model_name)
    embeddings = model.encode(texts, normalize_embeddings=True)
    return model, embeddings



In [16]:
# ==============================
# Clustering Utilities
# ==============================

def calculate_similarity_matrix(embeddings):
    """Calculate the cosine similarity between all pairs of embeddings."""
    return cosine_similarity(embeddings)

def create_neighborhoods(embeddings, avg_clusters_per_neighborhood=40):
    """Create initial neighborhoods using K-means clustering."""
    num_neighborhoods = max(1, round(len(embeddings) / avg_clusters_per_neighborhood))

    kmeans = KMeans(n_clusters=num_neighborhoods, random_state=42, n_init=10)
    neighborhood_labels = kmeans.fit_predict(embeddings)

    # Group indices by neighborhood
    neighborhoods = {i: [] for i in range(num_neighborhoods)}
    for idx, label in enumerate(neighborhood_labels):
        neighborhoods[label].append(idx)

    return neighborhoods
def expand_neighborhoods(neighborhoods, similarity_matrix, m_nearest=5):
    """
    Expand each neighborhood by including the M nearest clusters from other neighborhoods.

    Args:
        neighborhoods: Dictionary mapping neighborhood IDs to lists of cluster indices
        similarity_matrix: Matrix of cosine similarities between all clusters
        m_nearest: Number of nearest neighbors to include from other neighborhoods

    Returns:
        Dictionary of expanded neighborhoods
    """
    expanded_neighborhoods = {}

    for n_id, cluster_indices in neighborhoods.items():
        # Get indices of clusters not in this neighborhood
        other_clusters = list(set(range(len(similarity_matrix))) - set(cluster_indices))
        external_nearest = []

        # For each cluster in this neighborhood, find the M nearest from other neighborhoods
        for idx in cluster_indices:
            similarities = [(other_idx, similarity_matrix[idx][other_idx]) for other_idx in other_clusters]
            similarities.sort(key=lambda x: x[1], reverse=True)
            external_nearest.extend([s[0] for s in similarities[:m_nearest]])

        # Add the nearest external clusters to this neighborhood, removing duplicates
        expanded_neighborhoods[n_id] = list(set(cluster_indices + external_nearest))

    return expanded_neighborhoods

# ==============================
# Cluster Naming
# ==============================

def generate_cluster_name_description_openai(in_cluster_samples, out_cluster_samples, model="claude-3-5-sonnet-20240620"):
    """
    Uses Claude 3.5 to generate a name and description that defines
    the in-cluster samples while differentiating them from out-cluster samples.
    """
    prompt = (
        f"Below are two sets of text summaries:\n\n"
        f"**In-cluster summaries:**\n{in_cluster_samples}\n\n"
        f"**Out-cluster summaries:**\n{out_cluster_samples}\n\n"
        f"Generate a short but descriptive name and a summary that captures the theme of "
        f"the in-cluster summaries while clearly distinguishing them from the out-cluster summaries."
    )

    response = openai.ChatCompletion.create(
        model=model,
        messages=[{"role": "system", "content": prompt}],
        temperature=1
    )

    return response["choices"][0]["message"]["content"]

def generate_cluster_name_description(in_cluster_samples, out_cluster_samples, llm_pipeline):
    """
    Uses the HF model to generate a name and description that defines
    the in-cluster samples while differentiating them from out-cluster samples.

    Args:
        in_cluster_samples (list of str): 50 summaries from within the cluster.
        out_cluster_samples (list of str): 50 summaries from outside but near the cluster.

    Returns:
        dict: {"name": str, "description": str}
    """
    prompt = (
        f"Below are two sets of text summaries:\n\n"
        f"**In-cluster summaries:**\n{in_cluster_samples}\n\n"
        f"**Out-cluster summaries:**\n{out_cluster_samples}\n\n"
        f"Generate a short but descriptive name and a summary that captures the theme of "
        f"the in-cluster summaries while clearly distinguishing them from the out-cluster summaries."
    )

    # Define the messages for the pipeline
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt},
    ]

    # Define generation arguments
    generation_args = {
        "max_new_tokens": 250,
        "return_full_text": False,
        "temperature": 1.0,
        "do_sample": True,
    }

    # Generate the response
    output = llm_pipeline(messages, **generation_args)
    response = output[0]['generated_text']

    # Extract name and description from the response
    # Assuming the response format is consistent and can be split into name and description
    #name = "Generated Name"  # Placeholder for name extraction logic
    print(response)
    description = response

    # return {"name": name, "description": description}
    # return {"description": description}
    print(description)
    return description

def parse_cluster_info(cluster_info_text):
    """Parse the cluster name and description from Claude's response."""
    lines = cluster_info_text.split("\n")
    name = lines[0]
    description = "\n".join(lines[1:]) if len(lines) > 1 else ""
    return name, description

# ==============================
# Hierarchical Clustering
# ==============================

def perform_one_level_clustering(llm_pipeline, cluster_map, embedding_model, avg_clusters_per_neighborhood=40, m_nearest=5):
    """
    Perform one level of hierarchical clustering.

    Args:
        cluster_map: Current cluster mapping
        embedding_model: Model to create embeddings
        avg_clusters_per_neighborhood: Target number of clusters per neighborhood
        m_nearest: Number of nearest neighbors to include

    Returns:
        New cluster map with merged clusters
    """
    # Get cluster names and create embeddings
    cluster_texts = [cluster_map[i]["name"] for i in cluster_map.keys()]
    cluster_embeddings = embedding_model.encode(cluster_texts, normalize_embeddings=True)

    # Create initial neighborhoods using K-means
    neighborhoods = create_neighborhoods(cluster_embeddings, avg_clusters_per_neighborhood)

    # Calculate similarity between all clusters
    similarity_matrix = calculate_similarity_matrix(cluster_embeddings)

    # Expand neighborhoods to include nearby clusters
    expanded_neighborhoods = expand_neighborhoods(neighborhoods, similarity_matrix, m_nearest)

    # Create new cluster map by merging clusters in each neighborhood
    new_cluster_map = {}
    for n_id, cluster_indices in expanded_neighborhoods.items():
        # Get samples from inside the cluster and outside but nearby for contrast
        in_cluster_samples = [cluster_map[list(cluster_map.keys())[idx]]["name"] for idx in cluster_indices[:50]]
        out_cluster_samples = [cluster_map[list(cluster_map.keys())[idx]]["name"] for idx in cluster_indices[-50:]]

        # Generate cluster name and description
        cluster_info = generate_cluster_name_description(in_cluster_samples, out_cluster_samples, llm_pipeline)
        name, description = parse_cluster_info(cluster_info)

        # Create new cluster and add children
        new_cluster_map[n_id] = {"name": name, "description": description, "children": {}}
        for idx in cluster_indices:
            old_cluster_id = list(cluster_map.keys())[idx]
            new_cluster_map[n_id]["children"][old_cluster_id] = cluster_map[old_cluster_id]

    return new_cluster_map

def hierarchical_clustering(llm_pipeline, embeddings, texts, target_clusters, avg_clusters_per_neighborhood=40, m_nearest=5):
    """
    Performs hierarchical clustering with Claude-generated names and descriptions.

    Args:
        embeddings: Sentence embeddings of the dataset
        texts: Original conversation texts
        target_clusters: Desired number of top-level clusters
        avg_clusters_per_neighborhood: Target number of clusters per neighborhood
        m_nearest: Number of nearest neighbors to include

    Returns:
        Nested dictionary representing the cluster hierarchy
    """
    # Initialize model for creating embeddings
    model = SentenceTransformer("all-mpnet-base-v2")

    # Initialize with one document per cluster
    level = 0
    cluster_map = {i: {"name": texts[i], "children": {}} for i in range(len(texts))}

    # Keep clustering until we reach the target number of clusters
    while len(cluster_map) > target_clusters:
        level += 1
        print(f"\n🔄 Level {level}: Clustering {len(cluster_map)} clusters...")

        # Perform one level of clustering
        cluster_map = perform_one_level_clustering(
            llm_pipeline,
            cluster_map,
            model,
            avg_clusters_per_neighborhood,
            m_nearest
        )

    return cluster_map

# ==============================
# Exploration and Saving
# ==============================

def save_hierarchy(hierarchy, filename="cluster_hierarchy.json"):
    """Save the cluster hierarchy to a JSON file."""
    with open(filename, "w") as f:
        json.dump(hierarchy, f, indent=4)

def explore_clusters(cluster_hierarchy, depth=0):
    """Interactive exploration of the hierarchical clusters."""
    while True:
        print("\n" + "-" * (depth * 2) + " Available Clusters " + "-" * (depth * 2))
        for i, cluster_id in enumerate(cluster_hierarchy.keys()):
            print(f"{i}: {cluster_hierarchy[cluster_id]['name']} - {cluster_hierarchy[cluster_id].get('description', '')}")

        choice = input("\nEnter number to zoom in, 'b' to go back, or 'q' to quit: ")
        if choice == "q":
            break
        elif choice == "b":
            return
        elif choice.isdigit() and int(choice) in range(len(cluster_hierarchy)):
            selected_cluster = list(cluster_hierarchy.keys())[int(choice)]
            if cluster_hierarchy[selected_cluster]["children"]:
                explore_clusters(cluster_hierarchy[selected_cluster]["children"], depth + 1)
            else:
                print("\nNo further subclusters available.")

In [5]:
llm_path = "microsoft/Phi-3.5-mini-instruct"
llm = AutoModelForCausalLM.from_pretrained(
    llm_path,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(llm_path)
# Create a text generation pipeline
llm_pipeline = pipeline(
    "text-generation",
    model=llm,
    tokenizer=tokenizer,
)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [14]:
target_clusters=3
avg_clusters_per_neighborhood=4
m_nearest=5

In [7]:
texts = load_data(sample_size=200)
model, embeddings = create_embeddings(texts)


In [8]:
embeddings.shape

(200, 768)

In [17]:
final_hierarchy = hierarchical_clustering(
    llm_pipeline,
    embeddings,
    texts,
    target_clusters,
    avg_clusters_per_neighborhood,
    m_nearest
  )


🔄 Level 1: Clustering 200 clusters...
 Name: In-cluster Billing Dispute and Unexpected Charges Issues

Summary: The in-cluster summaries present a collection of consumer complaints focusing on two primary issues: Billing Disputes and Unanticipated Fees or Interest Charges. In particular, consumers express frustration with their financial institutions over errors and additional charges, such as incorrect credit reports, persistent billing after claiming to have resolved collection issues, and payments or interest rate hikes that defy their expectations or prior agreements. These complaints highlight the ongoing financial challenges consumers face when working with lenders or creditors, contrasting with potential out-cluster scenarios that may not focus on these same themes.
 Name: In-cluster Billing Dispute and Unexpected Charges Issues

Summary: The in-cluster summaries present a collection of consumer complaints focusing on two primary issues: Billing Disputes and Unanticipated Fees 

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.11 GiB. GPU 0 has a total capacity of 14.74 GiB of which 736.12 MiB is free. Process 109281 has 14.02 GiB memory in use. Of the allocated memory 12.12 GiB is allocated by PyTorch, and 1.78 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)